---
title: "Repository Search and Evidence"
draft: true
categories: [agents, workflows, langgraph, ai-engineering, search, retrieval, provenance]
---

LangGraph coordinates transitions; it does not make a search result relevant. This chapter builds the evidence contract underneath the graph and compares lexical, dense, structural, hybrid, and bounded agentic retrieval before using any result in a change plan.


## Retrieval preserves repository identity

The fixture catalog contains code, tests, configuration, documentation, and Git history. Each result preserves the repository revision, source location, content fingerprint, retrieval method, and rank. These fields let a reviewer reopen the exact material that supported a claim.


## Ingest a local repository

The same source contract can be built from a checkout instead of a hand-authored fixture. The indexer records Python symbols, infers test candidates, and binds the result to an explicit revision and content fingerprint.


In [1]:
from pathlib import Path

from change_planner.ingestion import ingest_repository


demo_root = Path(".tmp/change-planner-ingestion-demo")
(demo_root / "src").mkdir(parents=True, exist_ok=True)
(demo_root / "tests").mkdir(parents=True, exist_ok=True)
(demo_root / "src" / "cli.py").write_text(
    "def clear_outputs(notebook, dry_run=False):\n    return notebook if dry_run else []\n",
    encoding="utf-8",
)
(demo_root / "tests" / "test_cli.py").write_text(
    "from cli import clear_outputs\n\ndef test_clear_outputs():\n    assert clear_outputs(notebook, dry_run=True)\n",
    encoding="utf-8",
)
(demo_root / "README.md").write_text("A small CLI repository.\n", encoding="utf-8")

indexed = ingest_repository(demo_root, repository="fixture/ingestion-demo", revision="demo-1")
print(indexed.snapshot.model_dump())
print([(source.path, source.symbols, source.related_tests) for source in indexed.catalog.sources.values()])
assert indexed.catalog.get("fixture/ingestion-demo:src/cli.py").symbols == ["clear_outputs"]
assert indexed.snapshot.revision == "demo-1"


{'repository': 'fixture/ingestion-demo', 'revision': 'demo-1', 'index_id': 'sha256:ff3fe71a737eb7ae', 'source_fingerprint': 'sha256:ff3fe71a737eb7ae183568b201a723a46a1cb9df53f257002a64ec5300a36f17'}
[('README.md', [], []), ('src/cli.py', ['clear_outputs'], ['fixture/ingestion-demo:tests/test_cli.py']), ('tests/test_cli.py', ['test_clear_outputs'], [])]


The local index demonstrates the ingestion boundary. The canonical lesson fixture remains pinned so retrieval results and evaluation labels stay reproducible across machines.


In [1]:
from IPython.display import Markdown, display
from change_planner.fixtures import load_sources
from change_planner.retrieval import FixtureCatalog, hybrid_search

catalog = FixtureCatalog(load_sources())
evidence = hybrid_search(catalog, "dry run clear outputs", top_k=4)
rows = "\n".join(
    f"| `{item.path}` | {item.revision} | {', '.join(hit.method for hit in item.retrieval)} | {item.content_hash} |"
    for item in evidence
)
display(Markdown("| Path | Revision | Retrieval trace | Fingerprint |\n|---|---|---|---|\n" + rows))
assert evidence
assert all(item.revision == "8f2c1d" and item.content_hash.startswith("sha256:") for item in evidence)


| Path | Revision | Retrieval trace | Fingerprint |
|---|---|---|---|
| `src/change_cli/commands.py` | 8f2c1d | dense, lexical, structural | sha256:43dddac929c6a390 |
| `docs/commands.md` | 8f2c1d | dense, lexical, structural | sha256:513ae25a233a4139 |
| `tests/test_clear_outputs.py` | 8f2c1d | dense, lexical, structural | sha256:dfe005dae9622c85 |
| `git/8f2c1d.patch` | 8f2c1d | dense, lexical, structural | sha256:a8698eb71c69f60f |

The retrieval trace intentionally keeps multiple observations instead of collapsing them into one opaque score. Exact names often favor lexical search; related behavior can favor dense search; symbol and source relationships add structural candidates. A hybrid score is a ranking aid, not an impact proof.

## Evidence is narrower than inference

The next cell connects test evidence to implementation candidates. The relationship is initially a candidate and becomes verified only after the allowed check runs.


In [3]:
from change_planner.analysis import build_evidence_graph, test_links
from change_planner.schemas import Evidence


links = test_links(catalog, evidence)
graph = build_evidence_graph(catalog, evidence)
for link in links:
    print(link.model_dump())
print({
    "nodes": len(graph.nodes),
    "edges": len(graph.edges),
    "relations": [edge.relation for edge in graph.edges],
})
assert all(link.status == "candidate" for link in links)
assert any("test_clear_outputs" in link.test_evidence_id for link in links)
assert graph.revision == "8f2c1d"
assert all(edge.status == "candidate" for edge in graph.edges)
assert any(edge.relation == "tested_by" for edge in graph.edges)


{'id': 'test-link:fixture/change-cli@8f2c1d:tests/test_clear_outputs.py', 'test_evidence_id': 'fixture/change-cli@8f2c1d:tests/test_clear_outputs.py', 'target_evidence_ids': ['fixture/change-cli@8f2c1d:src/change_cli/commands.py'], 'relation': 'symbol_match', 'status': 'candidate'}
{'nodes': 4, 'edges': 5, 'relations': ['tested_by', 'changed_with', 'tested_by', 'changed_with', 'tested_by']}


An importing test or nearby symbol is evidence of a candidate relationship, not proof that the changed behavior is covered. The project keeps that distinction in `TestLink` and `RegressionHypothesis`. Chapter 04 routes stale indexes, missing evidence, and tool failures instead of hiding them behind an empty search result.
